# 02_02 — обучаемый Word2Vec baseline


Структура эксперимента
1. Унифицированно разбираются sem-метки CoNLL-U.
2. Данные делятся следующим образом:
   - train: десять файлов из bot/, cobald/, syntagrus/;
   - validation: ручной fiction;
   - test: ручные news, poetry, social, wiki.
3. Создаётся общий список семантических классов.
4. Для каждого токена строятся признаки:
   - Word2Vec-вектор самого слова;
   - агрегированный контекст в окне 5;
   - one-hot POS.
5. Для каждого класса отдельно обучается бинарный SGDClassifier:
   - loss='log_loss';
   - penalty='elasticnet';
   - alpha=1e-5;
   - l1_ratio=0.05;
   - class_weight='balanced';
   - max_iter=50.
6. Совокупность классификаторов образует One-vs-Rest multilabel-модель.
7. Порог каждого класса отдельно подбирается по precision-recall curve на fiction-validation.
8. Зафиксированные пороги применяются к четырём тестовым жанрам.
9. Признаки и отдельные классификаторы сохраняются по частям, поэтому эксперимент можно продолжать после прерывания.

---

In [2]:
%pip install -q gensim scipy pandas scikit-learn tqdm trackio


In [3]:
!git clone https://github.com/tertiadecima/phd_semantics.git /content/phd_semantics
%cd /content/phd_semantics


Cloning into '/content/phd_semantics'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 26 (delta 9), reused 21 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 119.78 KiB | 10.89 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/phd_semantics


## 1. Нормализация CoNLL-U и единое оценивание


### Подготовка и единое представление данных

Этот блок нормализует варианты записи семантических тегов, разбирает CoNLL-U, управляет промежуточным кэшем и задаёт единый протокол многометочного оценивания.


In [4]:
# Здесь собран общий слой чтения CoNLL-U: нормализует разные варианты `sem=[...]`, читает
# предложения потоково, атомарно сохраняет кэш и рассчитывает единые multilabel-метрики.
# При появлении нового формата MISC следует расширять `tags_from_misc`, сохраняя
# lowercase, удаление кавычек и дедупликацию.

import ast
import hashlib
import json
import os
import pickle
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEM_RE = re.compile(r"(?:^|\|)sem=(\[[^\]]*\]|[^|]+)")


def normalize_sem_tags(value):
    """Normalize sem=[a,b], sem=['a','b'] and sem=[\"a\", \"b\"]."""
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        raw_tags = value
    else:
        text = str(value).strip()
        try:
            # Шаг: пробуем разобрать значение MISC как настоящий Python-список с кавычками.
            parsed = ast.literal_eval(text)
            raw_tags = parsed if isinstance(parsed, (list, tuple, set)) else [parsed]
        except (ValueError, SyntaxError):
            raw_tags = text.strip("[]").split(",")
    return sorted({
        str(tag).strip().strip("'\"").lower()
        for tag in raw_tags
        if str(tag).strip().strip("'\"")
    })


def tags_from_misc(misc):
    # извлекаем только значение поля sem, не затрагивая остальные атрибуты MISC.
    match = SEM_RE.search(str(misc or ""))
    if not match:
        return [], False
    return normalize_sem_tags(match.group(1)), True


@dataclass
class Sentence:
    words: list[str]
    lemmas: list[str]
    upos: list[str]
    tags: list[list[str]]
    labeled: list[bool]
    source: str


def read_conllu(path):
    """Stream CoNLL-U; skip multiword and empty-node rows."""
    sentences, rows = [], []
    with Path(path).open(encoding="utf-8") as file:
        for line in file:
            line = line.rstrip("\n")
            if not line:
                if rows:
                    # завершаем накопленное предложение при встрече пустой строки CoNLL-U.
                    sentences.append(_rows_to_sentence(rows, str(path)))
                    rows = []
                continue
            if line.startswith("#"):
                continue
            columns = line.split("\t")
            if len(columns) == 10 and columns[0].isdigit():
                rows.append(columns)
    if rows:
        # завершаем накопленное предложение при встрече пустой строки CoNLL-U.
        sentences.append(_rows_to_sentence(rows, str(path)))
    return sentences


def _rows_to_sentence(rows, source):
    tags_and_masks = [tags_from_misc(row[9]) for row in rows]
    return Sentence(
        words=[row[1] for row in rows],
        lemmas=[row[2] if row[2] != "_" else row[1].lower() for row in rows],
        upos=[row[3] for row in rows],
        tags=[item[0] for item in tags_and_masks],
        labeled=[item[1] for item in tags_and_masks],
        source=source,
    )


def file_signature(paths):
    digest = hashlib.sha256()
    for path in sorted(map(Path, paths)):
        stat = path.stat()
        digest.update(f"{path}:{stat.st_size}:{stat.st_mtime_ns}".encode())
    return digest.hexdigest()[:16]


def atomic_pickle(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as file:
        # Шаг: сериализуем дорогой промежуточный результат для повторного использования.
        pickle.dump(value, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    # Шаг: атомарно заменяем checkpoint только после успешной полной записи.
    temporary.replace(path)


def load_records(paths, cache_dir, name):
    """Load and checkpoint every source file independently."""
    paths = list(map(Path, paths))
    file_cache_dir = Path(cache_dir) / "parsed_conllu"
    file_cache_dir.mkdir(parents=True, exist_ok=True)
    records = []
    progress = tqdm(paths, desc=f"Чтение {name}", unit="файл")
    for path in progress:
        cache = file_cache_dir / f"{path.stem}_{file_signature([path])}.pkl"
        file_records = None
        if cache.exists():
            try:
                with cache.open("rb") as file:
                    # Шаг: восстанавливаем уже разобранный файл, чтобы не повторять длительный парсинг.
                    file_records = pickle.load(file)
                status = "cache"
            except Exception:
                file_records = None
        if file_records is None:
            # Шаг: читаем новый файл и оставляем только предложения с sem-разметкой.
            file_records = [sentence for sentence in read_conllu(path)
                            if any(sentence.labeled)]
            atomic_pickle(file_records, cache)
            status = "saved"
        records.extend(file_records)
        progress.set_postfix(file=path.name[:24], status=status,
                             sentences=f"{len(records):,}")
    progress.close()
    print(f"{name}: {len(records):,} размеченных предложений")
    return records

def collect_classes(groups):
    # Шаг: строим единый детерминированный порядок классов для всех матриц и моделей.
    classes = sorted({
        tag
        for records in groups
        for sentence in records
        for tags, labeled in zip(sentence.tags, sentence.labeled)
        if labeled
        for tag in tags
    })
    if not classes:
        raise ValueError("Не найдено ни одной метки sem")
    return classes


def label_statistics(records, classes):
    counts = Counter(
        tag for sentence in records
        for tags, labeled in zip(sentence.tags, sentence.labeled)
        if labeled for tag in tags
    )
    return pd.DataFrame({"tag": classes, "support": [counts[tag] for tag in classes]}) \
        .sort_values("support", ascending=False).reset_index(drop=True)


def tune_thresholds(y_true, probabilities, min_support=2):
    from sklearn.metrics import precision_recall_curve
    thresholds = np.full(y_true.shape[1], 0.5, dtype=np.float32)
    for column in range(y_true.shape[1]):
        if y_true[:, column].sum() < min_support:
            continue
        # Шаг: перебираем возможные пороги конкретного класса по validation-разметке.
        precision, recall, candidates = precision_recall_curve(
            y_true[:, column], probabilities[:, column]
        )
        if not len(candidates):
            continue
        f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(
            precision[:-1] + recall[:-1], 1e-12
        )
        thresholds[column] = candidates[int(np.nanargmax(f1))]
    return thresholds


def multilabel_metrics(y_true, probabilities, thresholds, classes, split):
    from sklearn.metrics import precision_recall_fscore_support, accuracy_score, hamming_loss
    # Шаг: превращаем вероятности в multilabel-предсказания с отдельным порогом каждого тега.
    predicted = (probabilities >= thresholds).astype(np.int8)
    micro = precision_recall_fscore_support(
        y_true, predicted, average="micro", zero_division=0
    )
    supported = y_true.sum(axis=0) > 0
    macro = precision_recall_fscore_support(
        y_true[:, supported], predicted[:, supported], average="macro", zero_division=0
    )
    per_class = precision_recall_fscore_support(
        y_true, predicted, average=None, zero_division=0
    )
    summary = {
        "split": split,
        "tokens": int(len(y_true)),
        "labels": int(y_true.sum()),
        "micro_precision": micro[0], "micro_recall": micro[1], "micro_f1": micro[2],
        "macro_precision_supported": macro[0],
        "macro_recall_supported": macro[1],
        "macro_f1_supported": macro[2],
        "exact_match": accuracy_score(y_true, predicted),
        "hamming_loss": hamming_loss(y_true, predicted),
    }
    details = pd.DataFrame({
        "tag": classes, "precision": per_class[0], "recall": per_class[1],
        "f1": per_class[2], "support": per_class[3], "threshold": thresholds,
    }).sort_values("support", ascending=False)
    return summary, details


## 2. Пути и разделение данных


In [5]:
# В Colab исходные данные и модели читаются с Google Drive.
# Результаты, Trackio и checkpoints сохраняются локально внутри клонированного git-проекта.
import importlib.util
import os
from pathlib import Path


IS_COLAB = (importlib.util.find_spec('google') is not None and
             importlib.util.find_spec('google.colab') is not None)


def find_project_root(start=None):
    """Находит корень локально клонированного git-репозитория."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError(
        'Корень git-репозитория не найден. В Colab сначала клонируйте проект в /content '
        'и перейдите в его папку командой %cd.'
    )


PROJECT_ROOT = find_project_root()
if IS_COLAB:
    from google.colab import drive
    # Google Drive используется только как постоянное хранилище входных данных и моделей.
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Аспирантура')
else:
    # Вне Colab структура data/models берётся из локального git-проекта.
    DRIVE_ROOT = PROJECT_ROOT

# Поддерживаются обе структуры: DRIVE_ROOT/data и старый вариант DRIVE_ROOT/conllu.
DATA_ROOT = DRIVE_ROOT / 'data' if (DRIVE_ROOT / 'data').exists() else DRIVE_ROOT
MODELS_ROOT = DRIVE_ROOT / 'models' if (DRIVE_ROOT / 'models').exists() else DRIVE_ROOT

# Все изменяемые и промежуточные файлы остаются на локальном диске runtime.
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
CACHE_ROOT = OUTPUT_ROOT / 'experiment_cache'
LOCAL_CHECKPOINT_ROOT = OUTPUT_ROOT / 'checkpoints'
TRACKIO_ROOT = OUTPUT_ROOT / 'trackio'
for directory in (OUTPUT_ROOT, CACHE_ROOT, LOCAL_CHECKPOINT_ROOT, TRACKIO_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
os.environ['TRACKIO_DIR'] = str(TRACKIO_ROOT)

print('Режим:', 'Colab + Google Drive' if IS_COLAB else 'локальный запуск')
print('Git-проект и checkpoints:', PROJECT_ROOT)
print('Данные:', DATA_ROOT)
print('Модели:', MODELS_ROOT)
print('Checkpoints:', LOCAL_CHECKPOINT_ROOT)

MODEL_CANDIDATES = [
    MODELS_ROOT / 'main_POS_V300_W5_C5/main_POS_V300_W5_C5.model',
    MODELS_ROOT / 'main_POS_V300_W5_C5.model',
    DRIVE_ROOT / 'main_POS_V300_W5_C5.model',
]
MODEL_PATH = next((path for path in MODEL_CANDIDATES if path.exists()), MODEL_CANDIDATES[0])
MODEL_VECTOR_PATH = MODEL_PATH.with_name(MODEL_PATH.name + '.wv.vectors.npy')
MODEL_NEGATIVE_PATH = MODEL_PATH.with_name(MODEL_PATH.name + '.syn1neg.npy')
missing_model_files = [path for path in (MODEL_PATH, MODEL_VECTOR_PATH, MODEL_NEGATIVE_PATH)
                       if not path.exists()]
if missing_model_files:
    raise FileNotFoundError(
        'На Google Drive не найдены файлы Word2Vec-модели:\n' +
        '\n'.join(map(str, missing_model_files))
    )

CONLLU_CANDIDATES = [DATA_ROOT / 'conllu', DRIVE_ROOT / 'conllu']
CONLLU_ROOT = next((path for path in CONLLU_CANDIDATES if path.exists()),
                   CONLLU_CANDIDATES[0])

# Автоматически размеченные корпуса используются только для обучения.
TRAIN_FILES = sorted([
    *CONLLU_ROOT.glob('bot/*.conllu'),
    *CONLLU_ROOT.glob('cobald/*.conllu'),
    *CONLLU_ROOT.glob('syntagrus/*.conllu'),
])
HAND_ROOT = CONLLU_ROOT / 'handlabeled'
VALID_FILE = HAND_ROOT / 'GramEval2020-test-fiction.conllu'
TEST_FILES = {
    genre: HAND_ROOT / f'GramEval2020-test-{genre}.conllu'
    for genre in ('news', 'poetry', 'social', 'wiki')
}
missing_corpora = [path for path in [VALID_FILE, *TEST_FILES.values()] if not path.exists()]
if not TRAIN_FILES:
    missing_corpora.append(CONLLU_ROOT)
if missing_corpora:
    raise FileNotFoundError(
        'На Google Drive не найдены корпуса:\n' + '\n'.join(map(str, missing_corpora))
    )

DATA_SPLITS = {
    'train': [str(path) for path in TRAIN_FILES],
    'validation': [str(VALID_FILE)],
    'test': {genre: str(path) for genre, path in TEST_FILES.items()},
}
print(f'Обучающих файлов: {len(TRAIN_FILES)}')
print('Validation:', VALID_FILE.name)
print('Test:', ', '.join(path.name for path in TEST_FILES.values()))


Mounted at /content/drive
Режим: Colab + Google Drive
Git-проект и checkpoints: /content/phd_semantics
Данные: /content/drive/MyDrive/Colab Notebooks/Аспирантура
Модели: /content/drive/MyDrive/Colab Notebooks/Аспирантура
Checkpoints: /content/phd_semantics/outputs/checkpoints
Обучающих файлов: 10
Validation: GramEval2020-test-fiction.conllu
Test: GramEval2020-test-news.conllu, GramEval2020-test-poetry.conllu, GramEval2020-test-social.conllu, GramEval2020-test-wiki.conllu


## 3. Загрузка корпусов и канонический набор классов


In [6]:
# Здесь загружаются размеченные предложения из файлов или кэша, а затем строится
# канонический набор классов по всем используемым меткам. Тестовые тексты участвуют только в
# определении имён классов.

# загружаем train с восстановлением готовых файлов из локального кэша
(CACHE_ROOT / 'data_splits.json').write_text(
    json.dumps(DATA_SPLITS, ensure_ascii=False, indent=2), encoding='utf-8'
)
train_records = load_records(TRAIN_FILES, CACHE_ROOT, 'train_all')

# загружаем ручной validation-набор, используемый только для настройки порогов
valid_records = load_records([VALID_FILE], CACHE_ROOT, 'valid_fiction')

# создаём отдельные ручные тесты по жанрам без смешивания с обучением
test_records = {
    genre: load_records([path], CACHE_ROOT, f'test_{genre}')
    for genre, path in TEST_FILES.items()
}

# Канонический набор строится по ВСЕМ используемым меткам, включая ручные тесты.
# Сами тестовые предложения при этом никогда не используются для обучения.
# Шаг: фиксируем полный канонический список меток до построения выходных слоёв.
CLASSES = collect_classes([
    train_records, valid_records, *test_records.values()
])
TAG_TO_ID = {tag: index for index, tag in enumerate(CLASSES)}
(CACHE_ROOT / 'canonical_classes.json').write_text(
    json.dumps(CLASSES, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(f'Канонических классов: {len(CLASSES)}')
print(CLASSES)
display(label_statistics(train_records, CLASSES).head(30))


Чтение train_all:   0%|          | 0/10 [00:00<?, ?файл/s]

train_all: 83,980 размеченных предложений


Чтение valid_fiction:   0%|          | 0/1 [00:00<?, ?файл/s]

valid_fiction: 80 размеченных предложений


Чтение test_news:   0%|          | 0/1 [00:00<?, ?файл/s]

...

Чтение test_social:   0%|          | 0/1 [00:00<?, ?файл/s]

test_social: 109 размеченных предложений


Чтение test_wiki:   0%|          | 0/1 [00:00<?, ?файл/s]

test_wiki: 73 размеченных предложений
Канонических классов: 84
['abstr', 'age', 'animal', 'appear', 'art', 'artwork', 'be', 'behav', 'berry', 'changest', 'cin', 'class', 'cloth', 'color', 'concr', 'constr', 'contact', 'degr', 'degree', 'device', 'dir', 'disappear', 'disease', 'dish', 'dist', 'doc', 'dram', 'emot', 'etn', 'ev', 'famn', 'food', 'form', 'fruit', 'furn', 'grain', 'group', 'hum', 'humq', 'impact', 'instr', 'inter', 'kin', 'light', 'max', 'ment', 'min', 'move', 'mus', 'neg', 'org', 'part', 'patrn', 'perc', 'physiol', 'physq', 'place', 'plant', 'posit', 'poss', 'prof', 'propn', 'psych', 'put', 'qtm', 'quant', 'set', 'size', 'sound', 'speech', 'speed', 'sport', 'stuff', 'taste', 'temper', 'text', 'time', 'tool', 'topon', 'transp', 'unit', 'weapon', 'weather', 'weight']


,tag,support
0,abstr,78110
1,concr,69425
2,hum,33579
3,time,24967
4,place,18336
5,ev,13626
6,org,12081
7,degr,11700
8,topon,11146
9,max,10773


## 4. Word2Vec-признаки: слово, контекст и POS


### Построение Word2Vec-признаков

Для каждого размеченного слова объединяются его вектор, усреднённый локальный контекст и часть речи. Пакетное сохранение позволяет продолжать подготовку признаков после обрыва соединения.


In [7]:
# Здесь загружается модель НКРЯ и строятся признаки размеченного слова: его Word2Vec-вектор,
# средний вектор локального контекста и one-hot часть речи. Признаки сохраняются пакетами и
# восстанавливаются после перезапуска.
# Можно менять размер `window` и `sentences_per_checkpoint`. При замене Word2Vec-модели
# следует очистить старый кэш признаков, поскольку размерность и словарь модели могли измениться.

from gensim.models import Word2Vec
from scipy import sparse
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import SGDClassifier

# Шаг: загружаем модель НКРЯ; все последующие ключи должны иметь формат лемма_POS.
word2vec = Word2Vec.load(str(MODEL_PATH)).wv

POS_MAP = {
    'NOUN':'S', 'PROPN':'S', 'ADJ':'A', 'VERB':'V', 'AUX':'V',
    'ADV':'ADV', 'ADVPRO':'ADV', 'PRON':'SPRO', 'DET':'APRO',
    'NUM':'NUM', 'ADP':'PR', 'CCONJ':'CONJ', 'SCONJ':'CONJ',
    'PART':'PART', 'INTJ':'INTJ',
}
POS_CLASSES = sorted({pos for records in [train_records, valid_records, *test_records.values()]
                      for sentence in records for pos in sentence.upos})
POS_TO_ID = {pos: i for i, pos in enumerate(POS_CLASSES)}


def word_vector(lemma, upos):
    key = f"{lemma.lower()}_{POS_MAP.get(upos, upos)}"
    return word2vec[key] if key in word2vec else None


def build_features(records, classes, window=5, show_progress=True):
    tag_to_id = {tag: i for i, tag in enumerate(classes)}
    dimension = word2vec.vector_size
    features, targets = [], []
    iterator = tqdm(records, desc='Word2Vec-признаки', unit='предл.') if show_progress else records
    for sentence in iterator:
        # Шаг: заранее извлекаем векторы слов предложения, сохраняя None для OOV.
        vectors = [word_vector(lemma, pos)
                   for lemma, pos in zip(sentence.lemmas, sentence.upos)]
        for index, (tags, labeled) in enumerate(zip(sentence.tags, sentence.labeled)):
            if not labeled:
                continue
            target = vectors[index]
            # Шаг: собираем известные Word2Vec-векторы соседей, исключая целевое слово.
            context = [vectors[j] for j in range(max(0,index-window),
                       min(len(vectors),index+window+1))
                       if j != index and vectors[j] is not None]
            target_vec = target if target is not None else np.zeros(dimension)
            context_vec = np.mean(context, axis=0) if context else np.zeros(dimension)
            pos_vec = np.zeros(len(POS_CLASSES), dtype=np.float32)
            pos_vec[POS_TO_ID[sentence.upos[index]]] = 1.0
            # Шаг: объединяем целевое слово, контекст и POS в один обучающий вектор.
            features.append(np.concatenate([target_vec, context_vec, pos_vec]))
            row = np.zeros(len(classes), dtype=np.int8)
            for tag in tags:
                row[tag_to_id[tag]] = 1
            targets.append(row)
    return np.asarray(features, dtype=np.float32), np.asarray(targets, dtype=np.int8)


def cached_features(records, name, sentences_per_checkpoint=2000):
    class_signature = hashlib.sha256('\0'.join(CLASSES).encode()).hexdigest()[:12]
    sources = sorted({sentence.source for sentence in records})
    source_signature = file_signature(sources)
    job_dir = LOCAL_CHECKPOINT_ROOT / '02_02_features' / (
        f'{name}_{class_signature}_{source_signature}'
    )
    job_dir.mkdir(parents=True, exist_ok=True)
    x_parts, y_parts = [], []
    progress = tqdm(total=len(records), desc=f'Признаки {name}', unit='предл.')
    for start in range(0, len(records), sentences_per_checkpoint):
        end = min(start + sentences_per_checkpoint, len(records))
        checkpoint = job_dir / f'{start:07d}_{end:07d}.npz'
        loaded = False
        if checkpoint.exists():
            try:
                with np.load(checkpoint) as saved:
                    x_part, y_part = saved['x'], saved['y']
                loaded = True
            except Exception:
                loaded = False
        if not loaded:
            x_part, y_part = build_features(records[start:end], CLASSES,
                                            show_progress=False)
            temporary = checkpoint.with_suffix('.tmp.npz')
            # Шаг: сохраняем очередную часть признаков; готовые части будут пропущены при перезапуске.
            np.savez_compressed(temporary, x=x_part, y=y_part)
            temporary.replace(checkpoint)
        x_parts.append(x_part)
        y_parts.append(y_part)
        progress.update(end - start)
        progress.set_postfix(status='cache' if loaded else 'saved',
                             checkpoint=checkpoint.name)
    progress.close()
    return np.concatenate(x_parts), np.concatenate(y_parts)

x_train, y_train = cached_features(train_records, 'train')
x_valid, y_valid = cached_features(valid_records, 'valid_fiction')
print('Train:', x_train.shape, y_train.shape)


Признаки train:   0%|          | 0/83980 [00:00<?, ?предл./s]

Признаки valid_fiction:   0%|          | 0/80 [00:00<?, ?предл./s]

Train: (264592, 628) (264592, 84)


## 5. Обучение One-vs-Rest классификатора


### Обучение классификаторов семантических тегов

Для каждого тега обучается независимый бинарный классификатор. Состояние сохраняется после каждого класса, а классы без вариативности обрабатываются константной моделью.


In [8]:
# Здесь обучается отдельный логистический классификатор для каждого тега
# После каждого класса список моделей сохраняется атомарно; редкие или константные классы 
# обрабатываются `DummyClassifier`.

# Можно настраивать `alpha`, `l1_ratio`, `max_iter` и `tol`. `early_stopping` оставлен
# выключенным, так как внутренний stratified split падает на классах с одним положительным
# примером.

import trackio

# Каждый запуск получает отдельную запись в трекере с параметрами классификатора и подготовки признаков.
trackio.init(
    project='phd_semantics',
    name='02_02_word2vec_supervised',
    group='word2vec_supervised',
    config={
        'model': str(MODEL_PATH.relative_to(DRIVE_ROOT)),
        'context_window': 5,
        'sentences_per_checkpoint': 2000,
        'loss': 'log_loss',
        'penalty': 'elasticnet',
        'alpha': 1e-5,
        'l1_ratio': 0.05,
        'class_weight': 'balanced',
        'max_iter': 50,
        'tolerance': 1e-3,
        'random_seed': 42,
        'classes': len(CLASSES),
        'train_sentences': len(train_records),
    },
)


from sklearn.dummy import DummyClassifier

RUN_CHECKPOINT_DIR = LOCAL_CHECKPOINT_ROOT / '02_01_word2vec'
RUN_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FILE = RUN_CHECKPOINT_DIR / 'ovr_by_class.pkl'


class CheckpointedOVR:
    def __init__(self, estimators):
        self.estimators_ = estimators

    def predict_proba(self, x):
        columns = []
        for estimator in tqdm(self.estimators_, desc='Вероятности', unit='класс'):
            proba = estimator.predict_proba(x)
            # DummyClassifier for a constant class may expose a single column.
            if proba.shape[1] == 1:
                positive = np.full(len(x), float(estimator.classes_[0] == 1))
            else:
                positive_index = int(np.where(estimator.classes_ == 1)[0][0])
                positive = proba[:, positive_index]
            columns.append(positive)
        return np.column_stack(columns)


estimators = [None] * len(CLASSES)
if MODEL_FILE.exists():
    try:
        with MODEL_FILE.open('rb') as file:
            # восстанавливаем ранее рассчитанный объект из доверенного checkpoint
            saved_model = pickle.load(file)
        if saved_model.get('classes') == CLASSES:
            estimators = saved_model['estimators']
            print(f"Продолжение: готово {sum(e is not None for e in estimators)}/{len(CLASSES)} классов")
    except Exception as error:
        print('Checkpoint не прочитан, обучение начнётся заново:', error)

progress = tqdm(total=len(CLASSES),
                initial=sum(e is not None for e in estimators),
                desc='Обучение OVR', unit='класс')
for column, tag in enumerate(CLASSES):
    if estimators[column] is not None:
        continue
    target = y_train[:, column]
    # обрабатываем класс без вариативности константной моделью вместо SGD
    if np.unique(target).size == 1:
        estimator = DummyClassifier(strategy='constant', constant=int(target[0]))
    else:
        # создаём независимый бинарный классификатор (принадлежит / не принадлежит) текущего семантического тега
        estimator = SGDClassifier(
            loss='log_loss', penalty='elasticnet', alpha=1e-5,
            l1_ratio=0.05, class_weight='balanced', max_iter=50,
            early_stopping=False, tol=1e-3,
            random_state=42 + column, n_jobs=1,
        )
    # обучаем один OVR-классификатор на общем наборе признаков
    estimator.fit(x_train, target)
    estimators[column] = estimator
    atomic_pickle({'estimators': estimators, 'classes': CLASSES}, MODEL_FILE)
    progress.update(1)
    progress.set_postfix(tag=tag, positives=int(target.sum()), checkpoint='saved')
    # В журнал попадает число уже обученных и сохранённых бинарных моделей.
    trackio.log({'training/classes_completed': column + 1, 'training/class_positives': int(target.sum())})
progress.close()

classifier = CheckpointedOVR(estimators)
# получаем вероятности всех тегов для последующей пороговой калибровки
p_valid = classifier.predict_proba(x_valid)
THRESHOLDS = tune_thresholds(y_valid, p_valid)
np.save(RUN_CHECKPOINT_DIR / 'thresholds.npy', THRESHOLDS)


* Trackio project initialized: phd_semantics
* Trackio metrics logged to: /content/phd_semantics/outputs/trackio
* psutil detected, enabling automatic CPU/system metrics logging


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


* Created new run: 02_02_word2vec_supervised


Обучение OVR:   0%|          | 0/84 [00:00<?, ?класс/s]

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_stochastic_gradient.py:738: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_stochastic_gradient.py:738: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_stochastic_gradient.py:738: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_stochastic_gradient.py:738: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_stochastic

Вероятности:   0%|          | 0/84 [00:00<?, ?класс/s]

## 6. Per-class thresholds и жанровая проверка


### Проверка Word2Vec-классификатора

Вероятности преобразуются в наборы тегов с порогами, настроенными на validation, после чего рассчитываются одинаковые метрики для всех ручных жанров.


In [9]:
# Вычисляем вероятности Word2Vec-классификатора, применяем пороги, настроенные только
# на validation, и формирует сопоставимые отчёты для всех ручных жанров.

summaries, detail_tables = [], {}
summary, details = multilabel_metrics(
    y_valid, p_valid, THRESHOLDS, CLASSES, 'validation_fiction'
)
summaries.append(summary)
detail_tables['validation_fiction'] = details

for genre, records in test_records.items():
    x_test, y_test = cached_features(records, f'test_{genre}')
    # Шаг: получаем вероятности всех тегов для последующей пороговой калибровки.
    probabilities = classifier.predict_proba(x_test)
    summary, details = multilabel_metrics(
        y_test, probabilities, THRESHOLDS, CLASSES, f'test_{genre}'
    )
    summaries.append(summary)
    detail_tables[genre] = details

summary_table = pd.DataFrame(summaries)
display(summary_table)
display(detail_tables['validation_fiction'].head(40))
summary_table.to_csv(CACHE_ROOT / '02_02_metrics.csv', index=False)

# Записываем итоговые метрики каждого split одним шагом, чтобы их было удобно сравнивать.
trackio.log({
    f"{row['split']}/{metric}": float(row[metric])
    for row in summaries
    for metric in ('micro_precision', 'micro_recall', 'micro_f1',
                   'macro_f1_supported', 'exact_match', 'hamming_loss')
})
# Завершаем запуск только после сохранения итоговой таблицы.
trackio.finish()


Признаки test_news:   0%|          | 0/86 [00:00<?, ?предл./s]

Вероятности:   0%|          | 0/84 [00:00<?, ?класс/s]

Признаки test_poetry:   0%|          | 0/77 [00:00<?, ?предл./s]

Вероятности:   0%|          | 0/84 [00:00<?, ?класс/s]

Признаки test_social:   0%|          | 0/109 [00:00<?, ?предл./s]

...

Признаки test_wiki:   0%|          | 0/73 [00:00<?, ?предл./s]

Вероятности:   0%|          | 0/84 [00:00<?, ?класс/s]

,split,tokens,labels,micro_precision,micro_recall,micro_f1,macro_precision_supported,macro_recall_supported,macro_f1_supported,exact_match,hamming_loss
0,validation_fiction,512,949,0.225394,0.798736,0.351577,0.398869,0.805458,0.474543,0.0,0.065011
1,test_news,583,1144,0.226594,0.729895,0.345827,0.367942,0.692133,0.424290,0.0,0.064506
2,test_poetry,490,892,0.186203,0.744395,0.297891,0.280529,0.701105,0.345981,0.0,0.076045
3,test_social,455,842,0.184399,0.701900,0.292068,0.352603,0.773465,0.408936,0.0,0.074961
4,test_wiki,673,1213,0.202422,0.730420,0.316995,0.275447,0.544382,0.321146,0.0,0.067537


,tag,precision,recall,f1,support,threshold
14,concr,0.923497,0.908602,0.915989,186,7.974261e-01
37,hum,0.567901,0.878981,0.690000,157,3.546525e-06
0,abstr,0.861538,0.708861,0.777778,79,6.833193e-01
76,time,0.968750,0.794872,0.873239,39,9.932163e-01
51,part,0.577778,0.866667,0.693333,30,8.704664e-08
56,place,0.395349,0.566667,0.465753,30,4.751964e-02
47,move,0.894737,0.586207,0.708333,29,1.000000e+00
61,propn,0.636364,0.777778,0.700000,27,9.233198e-01
69,speech,0.888889,0.960000,0.923077,25,1.000000e+00
27,emot,0.452381,0.826087,0.584615,23,7.062410e-34


* Run finished. Uploading logs to Trackio (please wait...)
